In [4]:
# Import the section_properties module
from steel_lib.section_properties import create_aisc_section_selector

# Create the AISC selector
try:
    aisc = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')
    print("AISC database loaded successfully!")
    print(f"Sections: {aisc['database_info']['n_sections']}")
    print(f"Properties: {aisc['database_info']['properties_count']}")
except Exception as e:
    print(f"Error loading AISC database: {e}")

Loading AISC Shapes Database...
Loaded 2299 sections from AISC database
AISC database loaded successfully!
Sections: 2299
Properties: 69


In [6]:
# Test the fixed filtering function
results = aisc['select_by_properties']({
    "W_min": {"Max": 10.0,"Min": 8.0},
})
results
# print(f"Found {results['count']} sections")
# if results['count'] > 0:
#     print("First few sections:")
#     for i in range(min(3, results['count'])):
#         designation = results['designations'][i]
#         d = results['d'][i] 
#         Ix = results['Ix'][i]
#         print(f"  {designation}: d={d:.1f}, Ix={Ix:.0f}")

{'error': "Property 'W_min' not found. Available properties: ['W', 'A', 'd', 'ddet', 'Ht', 'h', 'OD', 'bf', 'bfdet', 'B']...",
 'count': 0}

In [37]:
import numpy as np
from numba import jit,njit, prange
import time
import forallpeople as si
from steel_lib.aisc_14th import optional_reporting_handcalc
si.environment("default")

# A computationally intensive function in pure Python
def sum_squared_diff_python(x, y):
    """
    Calculates the sum of squared differences between two arrays.
    """
    result = 0
    for i in range(len(x)):
        result += (x[i] - y[i])**2
    return result

# The same function accelerated with Numba's @jit decorator
@optional_reporting_handcalc(config_object=None,key=None, detailed=False)
def sum_squared_diff_numba(x, y):
    """
    Calculates the sum of squared differences between two arrays using Numba.
    The @jit(nopython=True) decorator compiles this function to machine code.
    """
    result = 0 
    for i in prange(len(x)):
        result += (x[i]  - y[i]  )**2
    return result

# The same function accelerated with Numba's @jit decorator
@jit(nopython=True)
def sum_squared_diff_numba_1(x, y):
    """
    Calculates the sum of squared differences between two arrays using Numba.
    The @jit(nopython=True) decorator compiles this function to machine code.
    """
    result = 0 
    for i in prange(len(x)):
        result += (x[i]  - y[i]  )**2
    return result
# Create two large NumPy arrays with random data
array1 = np.random.rand(10_000_000)
array2 = np.random.rand(10_000_000)

In [38]:
%%timeit
result_python = sum_squared_diff_python(array1, array2)

3 s ± 32.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
# %%timeit
# result_numba = sum_squared_diff_numba(array1, array2)

In [40]:
%%timeit
result_numba_1 = sum_squared_diff_numba_1(array1, array2)

12.3 ms ± 226 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [1]:
import numpy as np
import time
from steel_lib.aisc_14th import bolt_bearing, bolt_shear, block_shear, shear_yielding_rupture, flexural_14th, prying_action, flexural_15th
from numba import prange, jit

# Integer encoding for member types - MUCH faster than strings
MEMBER_TYPE_CODES = {
    'PL': 0,  # Plate
    'L': 1,   # Angle
}

# Set the number of test cases
num_cases = 10000000

# --- Generate random data for bolt_bearing ---
F_u_values = np.random.uniform(400, 600, size=num_cases).astype(np.float64)
d_bolt_values = np.random.uniform(16, 24, size=num_cases).astype(np.float64)
t_values = np.random.uniform(5, 20, size=num_cases).astype(np.float64)
P_u_values = np.random.uniform(100, 500, size=num_cases).astype(np.float64)
V_u_values = np.random.uniform(50, 300, size=num_cases).astype(np.float64)
S_r_values = np.random.uniform(50, 100, size=num_cases).astype(np.float64)
N_r_values = np.random.randint(2, 6, size=num_cases).astype(np.int64)
S_c_values = np.random.uniform(50, 100, size=num_cases).astype(np.float64)
N_c_values = np.random.randint(2, 4, size=num_cases).astype(np.int64)
L_ev_values = np.random.uniform(30, 60, size=num_cases).astype(np.float64)
L_eh_values = np.random.uniform(30, 60, size=num_cases).astype(np.float64)
dv_values = np.random.uniform(18, 26, size=num_cases).astype(np.float64)
dh_values = np.random.uniform(18, 26, size=num_cases).astype(np.float64)
c_values = np.random.uniform(1, 5, size=num_cases).astype(np.float64)
phi_values = np.full(num_cases, 0.75, dtype=np.float64)

# --- Generate random data for bolt_shear ---
F_nv_values = np.random.uniform(400, 600, size=num_cases).astype(np.float64)
A_bolt_values = np.random.uniform(0.5, 2.0, size=num_cases).astype(np.float64)
N_shear_planes_values = np.random.randint(1, 3, size=num_cases).astype(np.int64)

# --- Generate random data for block_shear ---
F_y_values = np.random.uniform(36, 65, size=num_cases).astype(np.float64)
coped_values = np.random.choice([True, False], size=num_cases)

# --- Generate random data for shear_yielding_rupture ---
d_b_values = np.random.uniform(0.5, 1.5, size=num_cases).astype(np.float64)
n_members_values = np.random.randint(1, 3, size=num_cases).astype(np.int64)
coped_syr_values = np.random.randint(0, 3, size=num_cases).astype(np.int64)

# --- Generate random data for flexural_14th ---
l_values = np.random.uniform(2.0, 5.0, size=num_cases).astype(np.float64)
k_a_values = np.random.uniform(0.5, 1.0, size=num_cases).astype(np.float64)
d_values = np.random.uniform(4.0, 8.0, size=num_cases).astype(np.float64)
e_override_values = np.random.uniform(0.1, 0.5, size=num_cases).astype(np.float64)

# --- Generate random data for prying_action ---
F_nt_values = np.random.uniform(80, 100, size=num_cases).astype(np.float64)
F_nv_values_prying = np.random.uniform(50, 60, size=num_cases).astype(np.float64)
n_bolts_values = np.random.randint(2, 6, size=num_cases).astype(np.int64)
N_t_values = np.random.randint(2, 6, size=num_cases).astype(np.int64)
L_values = np.random.uniform(2.5, 4.0, size=num_cases).astype(np.float64)  # Changed from L_osl_values to L_values
ga_values = np.random.uniform(3.0, 4.5, size=num_cases).astype(np.float64)
prying_values = np.random.choice([True, False], size=num_cases)

# --- Generate random data for flexural_15th ---
# OLD WAY (slow): member_type_values = np.random.choice(['PL', 'L'], size=num_cases)
# NEW WAY (fast): Use integer codes instead of strings
member_type_codes = np.random.randint(0, 2, size=num_cases, dtype=np.int32)  # 0='PL', 1='L'
a_values = np.random.uniform(8, 12, size=num_cases).astype(np.float64)
E_values = np.full(num_cases, 29000, dtype=np.float64)

# Array to store the results
results_bearing = np.zeros(num_cases, dtype=np.float64)
results_shear = np.zeros(num_cases, dtype=np.float64)
results_block = np.zeros(num_cases, dtype=np.float64)
results_syr = np.zeros(num_cases, dtype=np.float64)
results_flexural = np.zeros(num_cases, dtype=np.float64)
results_prying = np.zeros(num_cases, dtype=np.float64)
results_flexural_15th = np.zeros(num_cases, dtype=np.float64)

@jit(nopython=True, parallel=True,fastmath = True)
def test_bolt_strength_function(
    # Bearing args
    F_u_values, d_bolt_values, t_values, P_u_values, V_u_values, S_r_values, N_r_values, S_c_values, N_c_values, L_ev_values, L_eh_values, dv_values, dh_values, c_values,
    # Shear args
    F_nv_values, A_bolt_values, N_shear_planes_values,
    # Block shear args
    F_y_values, coped_values,
    # Shear yielding rupture args
    d_b_values, n_members_values, coped_syr_values,
    # Flexural args
    l_values, k_a_values, d_values, e_override_values,
    # Prying action args
    F_nt_values, F_nv_values_prying, n_bolts_values, N_t_values, L_values, ga_values, prying_values,
    # Flexural_15th args (using INTEGER CODES - FAST!)
    member_type_codes, a_values, E_values,
    # Common args
    phi_values, 
    # Result arrays
    results_bearing, results_shear, results_block, results_syr, results_flexural, results_prying, results_flexural_15th,
    # Other
    num_cases):
    for i in prange(num_cases):
        # For simplicity in this test, we'll set N_c to be 1 for some cases to test that branch
        # In a real scenario, you might want to test different logic branches more systematically.
        n_c = 1 if i % 2 == 0 else N_c_values[i]
        
        bearing_val = bolt_bearing(
            F_u=F_u_values[i],
            d_bolt=d_bolt_values[i],
            t=t_values[i],
            P_u=P_u_values[i],
            V_u=V_u_values[i],
            S_r=S_r_values[i],
            N_r=N_r_values[i],
            S_c=S_c_values[i],
            N_c=n_c,
            L_ev=L_ev_values[i],
            L_eh=L_eh_values[i],
            dv=dv_values[i],
            dh=dh_values[i],
            phi=phi_values[i],
            c=c_values[i]
        )
        results_bearing[i] = bearing_val

        shear_val = bolt_shear(
            F_nv=F_nv_values[i],
            A_bolt=A_bolt_values[i],
            N_shear_planes=N_shear_planes_values[i],
            phi=phi_values[i]
         )
        results_shear[i] = shear_val

        block_val = block_shear(
            P_u=P_u_values[i],
            V_u=V_u_values[i],
            F_y=F_y_values[i],
            F_u=F_u_values[i],
            t=t_values[i],
            N_r=N_r_values[i],
            S_r=S_r_values[i],
            N_c=n_c,
            S_c=S_c_values[i],
            L_ev=L_ev_values[i],
            L_eh=L_eh_values[i],
            d_v=dv_values[i],
            d_h=dh_values[i],
            phi=phi_values[i],
            coped=coped_values[i]
        )
        results_block[i] = block_val

        syr_val = shear_yielding_rupture(
            N_r=N_r_values[i],
            S_r=S_r_values[i],
            L_ev=L_ev_values[i],
            t=t_values[i],
            d_b=d_b_values[i],
            F_y=F_y_values[i],
            F_u=F_u_values[i],
            phi=phi_values[i],
            n_members=n_members_values[i],
            coped=coped_syr_values[i]
        )
        if syr_val is not None:
            results_syr[i] = syr_val
        else:
            results_syr[i] = np.nan

        flexural_val = flexural_14th(
            l=l_values[i],
            k_a=k_a_values[i],
            L_eh=L_eh_values[i],
            N_c=n_c,
            _S_c=S_c_values[i],
            t=t_values[i],
            N_r=N_r_values[i],
            S_r=S_r_values[i],
            d_b=d_b_values[i],
            d=d_values[i],
            F_y=F_y_values[i],
            F_u=F_u_values[i],
            e_override=e_override_values[i]
        )
        results_flexural[i] = flexural_val

        prying_val = prying_action(
            t=t_values[i],
            F_u=F_u_values[i],
            F_nv=F_nv_values_prying[i],
            F_nt=F_nt_values[i],
            N_t=N_t_values[i],
            n_bolts=n_bolts_values[i],
            L=L_values[i],  # Changed from L_osl to L
            L_eh=L_eh_values[i],
            L_ev=L_ev_values[i],
            d_bolt=d_bolt_values[i],
            dv=dv_values[i],
            S_r=S_r_values[i],
            ga=ga_values[i],
            V_u=V_u_values[i],
            P_u=P_u_values[i],
            prying=prying_values[i]
        )
        if prying_val is not None:
            results_prying[i] = prying_val
        else:
            results_prying[i] = np.nan

        member_type_code = member_type_codes[i]
        
        # Convert integer code to string for function call
        if member_type_code == 0:
            member_type_str = 'PL'
        else:
            member_type_str = 'L'
        
        flexural_15th_val = flexural_15th(
            member_type=member_type_str,
            a=a_values[i],
            t=t_values[i],
            F_y=F_y_values[i],
            E=E_values[i],
            d_b=d_b_values[i],  # Use d_b instead of d
            L_ev=L_ev_values[i],
            F_u=F_u_values[i],
            N_r=N_r_values[i],
            S_r=S_r_values[i],
            d_bolt=d_bolt_values[i]
        )
        if flexural_15th_val is not None:
            results_flexural_15th[i] = flexural_15th_val
        else:
            results_flexural_15th[i] = np.nan    
    
    return results_bearing, results_shear, results_block, results_syr, results_flexural, results_prying, results_flexural_15th

In [2]:
%%timeit
test_bolt_strength_function(
    F_u_values, d_bolt_values, t_values, P_u_values, V_u_values, S_r_values, N_r_values, S_c_values, N_c_values, L_ev_values, L_eh_values, dv_values, dh_values, c_values,
    F_nv_values, A_bolt_values, N_shear_planes_values,
    F_y_values, coped_values,
    d_b_values, n_members_values, coped_syr_values,
    l_values, k_a_values, d_values, e_override_values,
    F_nt_values, F_nv_values_prying, n_bolts_values, N_t_values, L_values, ga_values, prying_values,
    member_type_codes, a_values, E_values,
    phi_values,
    results_bearing, results_shear, results_block, results_syr, results_flexural, results_prying, results_flexural_15th,
    num_cases
)

459 ms ± 14.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%%timeit
test_bolt_strength_function_base(
    F_u_values, d_bolt_values, t_values, P_u_values, V_u_values, S_r_values, N_r_values, S_c_values, N_c_values, L_ev_values, L_eh_values, dv_values, dh_values, c_values,
    F_nv_values, A_bolt_values, N_shear_planes_values,
    F_y_values, coped_values,
    d_b_values, n_members_values, coped_syr_values,
    l_values, k_a_values, d_values, e_override_values,
    F_nt_values, F_nv_values_prying, n_bolts_values, N_t_values, L_values, ga_values, prying_values,
    member_type_codes, a_values, E_values,
    phi_values,
    results_bearing, results_shear, results_block, results_syr, results_flexural, results_prying, results_flexural_15th,
    num_cases
)

In [ ]:
from math import sin,cos,pi
from handcalcs.decorator import handcalc
import forallpeople as si
from numba import njit
from steel_lib.aisc_14th import bolt_bearing, block_shear, shear_yielding_rupture, tensile_yielding_rupture, prying_action, flexural_14th, flexural_15th
from numpy import atan,radians


ao1_angle_bearing_args = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.375,
    "P_u": 5,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.25,
    "dv": 0.938,
    "dh": 0.938,
    "phi": 0.75,
    "c": 4
}
ao1_beam_bearing_args = {
    "F_u": 65,
    "d_bolt": 7/8,
    "t": 0.25,
    "P_u": 5,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.75,
    "dv": 0.938,
    "dh": 0.938,
    "phi": 0.75
}
ao1_0sl_angle_bearing_args = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.375,
    "P_u": 0,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.25,
    "dv": 0.938,
    "dh": 0.938,
    "phi": 0.75,
    'c':4
}

a02_osl_bearing = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.5,
    "P_u": 0,
    "V_u": 70,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.428,
    "dv": 0.938,
    "dh": 0.938,
    "phi": 0.75,
    'c':4
}

a03_nsl_plate_bearing = {
    "F_u": 65,
    "d_bolt": 1+1/8,
    "t": 0.625,
    "P_u": 60,
    "V_u": 90,
    "S_r": 3,
    "N_r": 5,
    "S_c": 3,
    "N_c": 3,
    "L_ev": 1.5,
    "L_eh": 2,
    "dv":  1.25,
    "dh": 1.5,
    "phi": 0.75,
    'c':5.226
}

ao3_block_shear_plate = {
    "P_u": 60, 
    "V_u": 90, 
    "F_y": 50, 
    "F_u": 65, 
    "t": 0.625, 
    "N_r": 5, 
    "S_r": 3, 
    "N_c": 3, 
    "S_c": 3, 
    "L_ev": 1.5, 
    "L_eh": 2, 
    "d_hole": 1.5 + 1/16, 
    "phi": 0.75,
    "coped":True,
    "n_members": 2
}
ao1_nsl_block_shear_plate = {
    "P_u": 5, 
    "V_u": 30, 
    "F_y": 36, 
    "F_u": 58, 
    "t": 0.375, 
    "N_r": 4, 
    "S_r": 3, 
    "N_c": 1, 
    "S_c": 3, 
    "L_ev": 1.25, 
    "L_eh": 1.25, 
    "d_v": 0.938, 
    "d_h": 0.938,
    "phi": 0.75,
    "coped":False,
    "n_members": 2
}
# block_shear(**ao1_nsl_block_shear_plate)

ao1_shear_yielding_rupture_plate_nsl = {
    "N_r": 4,
    "S_r": 3,
    "L_ev": 1.25,
    "t": 0.375,
    "d_b": 7/8,
    "F_y": 36,
    "F_u": 58,
    "phi": 0.75,
    "n_members": 2,
    "coped": 2
}
ao1_tensile_yielding_rupture_plate_nsl = {
    "N_r": 4,
    "S_r": 3,
    "L_ev": 1.25,
    "t": 0.375,
    "d_b": 7/8,
    "F_y": 36,
    "F_u": 58,
    "phi": 0.75,
    "n_members": 2,
}

# flexural(l = 3.5,k_a = 3/4 ,L_eh = 1.25, N_c = 1, _S_c = 3, t = 0.375, N_r = 4, S_r = 3, d_b = 7/8, d = None, F_y = 36, F_u = 58, e_override=None)
# a01_prying_osl_angle = prying_action(F_nt = 90,F_nv = 54, n_bolts = 4,t = 0.375, F_u = 58, N_t = 4, L = 3, L_eh = 1.25, L_ev = 1.25, d_bolt = 7/8, dv = 0.938, S_r = 3, ga = 3.75, P_u = 5,V_u = 30, prying= True)
# a02_prying_osl_support = prying_action(F_nt = 90,F_nv = 54, n_bolts = 4,t = 0.355, F_u = 65, N_t = 4, L = 0, L_eh = 1.25, L_ev = 3, d_bolt = 7/8, dv = 0.938, S_r = 3, ga = 3.75, P_u = 5,V_u = 30, prying= True,bf = 7.5)

a03_1_plate_flexure = {
    "member_type": "PL",
    "a": 10,
    "t": 0.625,
    "F_y": 50,
    "E": 29000,
    "F_u": 65,
    "N_r":5,
    "S_r":3,
    "d_bolt":1.25,
    "L_ev":1.5,
    "F_u":65,
    "d_b": 18

}

a03_flexural_nsl = flexural_15th(**a03_1_plate_flexure)

In [14]:
%%render
xg = ((bf/2)*(tf)*(bf/4)*4 + (((d-2*tf)*tw/2)*tw/4)*2)/((bf*tf*2)+(d-2*tf)*tw)

<IPython.core.display.Latex object>

In [12]:
%%render
dasd = ((2*bf**2*tf)+(tw**2*(d-2*tf)))/((8*bf*tf)+(4*tw*(d-2*tf)))

<IPython.core.display.Latex object>

In [ ]:

%%render
asdasdasd = ((bf/2*tf*bf/4 * 2) +(( d - 2*tf) *( tw/2) * tw/4))/((bf/2 *tf * 2) + (d-2 *tf) * tw/2)

<IPython.core.display.Latex object>